# Taxonomic Relevance Evaluation

This notebook evaluates a problem in the current species scoring setup: the manual annotations often encode dataset relevance through coarse taxonomic group and richness information, while the extraction output often enumerates detailed taxa. That creates large numbers of false positives and false negatives even when the extraction is arguably useful for screening dataset relevance.

The working hypothesis tested here is that we should compare several parallel taxonomic views instead of relying on the raw `species` field alone:

- raw `species` strings with the existing enhanced matcher,
- derived `taxon_richness_counts` to compare count-bearing GT strings against enumerated predictions,
- derived `taxon_richness_group_keys` to compare count + normalized group labels,
- derived `gbif_keys` to collapse vernacular vs scientific naming.


## Questions

1. Which mismatch patterns are evaluation artifacts rather than extraction failures?
2. Does a count-focused derived field recover signal from cases like `73 weevil species` vs 73 enumerated taxa?
3. Does GBIF enrichment improve comparison for vernacular/scientific mismatches?
4. Which derived fields reflect dataset relevance better than the baseline `species` metric?


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from llm_metadata.schemas.data_paper import RunArtifact
from llm_metadata.schemas.fuster_features import DatasetFeatures
from llm_metadata.taxonomy_eval import (
    DEFAULT_TAXONOMY_FIELDS,
    build_ground_truth_by_id,
    enrich_indexed_models,
    evaluate_taxonomy_fields,
)


In [ ]:
RUN_GLOB = "*20260305_135009_dev_subset_pdf_file*.json"
GT_PATH = Path("data/dataset_092624_validated.xlsx")

candidates = sorted(Path("artifacts/runs").glob(RUN_GLOB))
RUN_ARTIFACT_PATH = candidates[0] if candidates else Path("artifacts/runs/20260305_135009_dev_subset_pdf_file.json")

if not RUN_ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f"Could not find run artifact. Update RUN_GLOB or RUN_ARTIFACT_PATH. Tried: {RUN_ARTIFACT_PATH}"
    )

artifact = RunArtifact.load_json(RUN_ARTIFACT_PATH)
allowed_ids = [record.gt_record_id for record in artifact.records]
true_by_id = build_ground_truth_by_id(GT_PATH, allowed_ids=allowed_ids)
pred_by_id = artifact.predictions_by_id(DatasetFeatures)

display(Markdown(f"**Run artifact:** `{RUN_ARTIFACT_PATH}`  \
+**Records loaded:** GT={len(true_by_id)} | Pred={len(pred_by_id)}`"))


## Proposed Solution Under Test

The proposal is not to replace the raw `species` field. It is to keep it, then derive additional taxonomic comparison views from the same `DatasetFeatures` objects:

- `parsed_species`: structured analysis view only,
- `taxon_richness_mentions`: structured count + group mentions,
- `taxon_richness_counts`: exact comparison on projected counts,
- `taxon_richness_group_keys`: exact comparison on projected `count|group` keys,
- `gbif_keys`: exact comparison on resolved taxon identifiers.

This keeps the extraction contract stable while making evaluation more aligned with the annotation intent.


In [ ]:
FIELDS = [
    "species",
    "taxon_richness_counts",
    "taxon_richness_group_keys",
    "gbif_keys",
]

# Set include_gbif=False if you want to avoid live GBIF lookups or rely only on cached calls.
report = evaluate_taxonomy_fields(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=FIELDS,
    include_gbif=True,
    gbif_confidence_threshold=80,
)

metrics_df = report.metrics_to_pandas().sort_values(["f1", "precision", "recall"], ascending=False)
metrics_df


In [ ]:
true_enriched = enrich_indexed_models(true_by_id, include_gbif=False)
pred_enriched = enrich_indexed_models(pred_by_id, include_gbif=False)
detail_df = report.to_pandas()

species_mismatches = detail_df[(detail_df["field"] == "species") & (~detail_df["match"])]
richness_mismatches = detail_df[(detail_df["field"] == "taxon_richness_counts") & (~detail_df["match"])]
group_mismatches = detail_df[(detail_df["field"] == "taxon_richness_group_keys") & (~detail_df["match"])]

display(Markdown("### Baseline species mismatches"))
display(species_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(20))

display(Markdown("### Count-based mismatches"))
display(richness_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(20))

display(Markdown("### Count+group mismatches"))
display(group_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(20))


In [ ]:
records = []
for record_id in sorted(set(true_enriched) | set(pred_enriched), key=int):
    true_model = true_enriched.get(record_id)
    pred_model = pred_enriched.get(record_id)
    records.append(
        {
            "record_id": record_id,
            "true_species": getattr(true_model, "species", None),
            "pred_species": getattr(pred_model, "species", None),
            "true_richness_mentions": getattr(true_model, "taxon_richness_mentions", None),
            "pred_richness_mentions": getattr(pred_model, "taxon_richness_mentions", None),
            "true_richness_counts": getattr(true_model, "taxon_richness_counts", None),
            "pred_richness_counts": getattr(pred_model, "taxon_richness_counts", None),
            "true_group_keys": getattr(true_model, "taxon_richness_group_keys", None),
            "pred_group_keys": getattr(pred_model, "taxon_richness_group_keys", None),
        }
    )

taxonomy_view_df = pd.DataFrame.from_records(records)
taxonomy_view_df.head(20)


## Discussion Guide

Use the outputs above to answer these points explicitly:

- Which baseline `species` mismatches become reasonable matches under `taxon_richness_counts`?
- Which cases remain unresolved because group identity is still too coarse or too specific?
- How often does `gbif_keys` solve scientific vs vernacular mismatch without helping count/group cases?
- Do the new fields better reflect dataset relevance, or are they masking real extraction errors?

Alternatives to assess if the derived fields are still insufficient:

- prompt the model to extract structured taxon objects directly,
- infer broader taxonomic groups from GBIF hierarchy for enumerated predictions,
- add a small manual adjudication subset to validate that the new metrics really track relevance.
